# Load Libraries and Data

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import AsinhNorm
from shapely.geometry import Point


#Load the cleaned data
clean_rad = pd.read_excel("clean_data_kp.xlsx")

#### Create geodataframe for geopandas

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

geometry = [Point(xy) for xy in zip(clean_rad["lon"], clean_rad["lat"])]
geo_rad = gpd.GeoDataFrame(clean_rad, geometry=geometry, crs="EPSG:4326")

### Define a constant colormap to use for all visuals

Default: `managua` 

`managua` is very good for highlighting all points on the graph.  
Change to `Reds` if you would rather only easily see extreme high values. All low values become very hard to see on the light background

In [ ]:
COLOR = "managua"

# Compare X-ray Detectors

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
axes = axes.flatten()

detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]

for i, detector in enumerate(detectors):
    world.plot(ax=axes[i], color="lightgrey")
    geo_rad.plot(
        ax=axes[i],
        markersize=1,
        alpha=0.5,
        column=detector,
        cmap=COLOR,
        legend=True
    )
    axes[i].set_title(f"Variable: {detector}")

fig.suptitle(f"Xray Detector Counts Per Second")
plt.show()

## Analysis of detector comparison

- Given the large energy threshold differences between the four detectors, they all have vastly different scales
- Despite their very different scales, they all appear to follow the same trends
    - Low values in the southern hemisphere
    - A medium intensity line over Canada
    - Highest values in Arctic Circle
- Observations become spottier in `xray2_ps` and `xray3_ps`, but the trends are still there

# Compare X-ray detectors with asinh scale

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), constrained_layout=True)
axes = axes.flatten()

detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]

for i, detector in enumerate(detectors):
    world.plot(ax=axes[i], color="lightgrey")
    norm = AsinhNorm(linear_width=1, vmin=geo_rad[detector].min(), vmax=geo_rad[detector].max())
    geo_rad.plot(
        ax=axes[i],
        markersize=1,
        alpha=0.5,
        column=detector,
        cmap=COLOR,
        norm=norm,
        legend=True
    )
    axes[i].set_title(f"Variable: {detector}")

fig.suptitle(f"Xray Detector Counts Per Second (asinh scale)")
plt.show()

## Analysis of asinh detector split
- With the nature of asinh being linear at low values, `xray3_ps` remains entirely unchanged under this scale
- X-rays detected in the Arctic Circle and SAA are especially highlighted in `xray0_ps` and `xray1_ps`
- Disparity between X-ray counts in the northern and southern hemispheres is still visible

# Compare X-ray detectors split by month

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 12), constrained_layout=True)

months = ["Jan", "Feb", "Mar", "Apr"]
detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]

for i, detector in enumerate(detectors):
    vmin = geo_rad[detector].min()
    vmax = geo_rad[detector].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for j, month in enumerate(months):
        subset = geo_rad[geo_rad["month"] == month]

        world.plot(ax=axes[i, j], color="lightgrey")
        subset.plot(
            ax=axes[i, j],
            markersize=1,
            alpha=0.5,
            column=detector,
            cmap=COLOR,
            norm=norm,
            legend=True

        )
        axes[i, j].set_title(f"{detector} in {month}")

fig.suptitle(f"X-rays Per Second by Detector and Month",  size='xx-large')
plt.show()

## Analysis of monthly splitting
- Splitting by month shows that all sensors follow similar trends on a monthly basis
- As seen in the previous analyses, X-rays are higheest in the Arctic Circle, though mostly in March and April
- March has the most data, but April has the highest percentage of high values
- The first high values picked up by `xray0_ps` were mostly over Asia and in February
- It might be worth splitting March by week to see if the increase in X-ray values is as correlated with time as it appears when split like this

# Split detectors by month and scale by asinh

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 12), constrained_layout=True)

months = ["Jan", "Feb", "Mar", "Apr"]
detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]

for i, detector in enumerate(detectors):
    norm = AsinhNorm(linear_width=1, vmin=geo_rad[detector].min(), vmax=geo_rad[detector].max())
    for j, month in enumerate(months):
        subset = geo_rad[geo_rad["month"] == month]

        world.plot(ax=axes[i, j], color="lightgrey")
        subset.plot(
            ax=axes[i, j],
            markersize=1,
            alpha=0.5,
            column=detector,
            cmap=COLOR,
            norm=norm,
            legend=True

        )
        axes[i, j].set_title(f"{detector} in {month}")

fig.suptitle(f"X-rays Per Second by Detector and Month (asinh scale)",  size='xx-large')
plt.show()

## Analysis of monthly splitting scaled by asinh
- X-rays are visibly higher in the SAA in January and February
- With this scale, there are small spikes in X-ray counts in Jaunary and February in the southern hemisphere that were not visible before
- X-rays are incredibly intense in April
- Trends are still holding under this scale

# Compare detectors with a line graph

## Graph the entire time frame

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
grouped = clean_rad.groupby("group")[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]].mean()

for column in grouped.columns:
    ax.plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second")

ax.set_title("X-rays Per Second For All Detectors", fontsize="xx-large")
ax.set_xlabel("Group Number")
ax.set_ylabel("Particles Per Second (Log)")

ax.annotate(
    text='Divergence',            # The text display label
    xy=(37, -2.5),                # (x, y) coordinates the arrow points to
    xytext=(-8, -9),              # (x, y) coordinates where the text sits
    arrowprops=dict(
        facecolor='black',        # Arrow fill color
        arrowstyle='->'           # Clean arrow head style
    ),
    fontsize=12,                  # Text sizing
    color='black'                 # Text color
)


ax.legend()
plt.show()

Peaks and valleys seem to be very similar for most groups. Let's split by month to get a better view

## Split by month

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
axes = axes.flatten()

months = ["Jan", "Feb", "Mar", "Apr"]

for i, month in enumerate(months):
    subset = clean_rad[clean_rad["month"] == month]
    grouped = subset.groupby("group")[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]].mean()

    for column in grouped.columns:
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second")

    axes[i].set_title(f"Month: {month}", fontsize="xx-large")
    axes[i].set_ylim(-10, 10)

fig.suptitle("Group averages for all xray detectors by month", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper left", fontsize="x-large")
plt.show()

## Split by month and invert results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(28, 12))

axes = axes.flatten()

for i, month in enumerate(months):
    subset = clean_rad[clean_rad["month"] == month]
    grouped = subset.groupby("group")[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]].mean()

    for column in grouped.columns:
        axes[i].plot(grouped.index, -1 * np.log(grouped[column]), label=f"{column[:-3]} per second")

    axes[i].set_title(f"Month: {month}", fontsize="xx-large")
    axes[i].set_ylim(-10, 10)

fig.suptitle("Group averages for all xray detectors by month (Mult by -1)", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", fontsize="xx-large")
plt.show()

I inverted the results because it feels easier to compare with `xray0_ps` on the bottom since it has the lowest energy threshold. All of the items in the graph are exactly the same but multiplied by -1

## Analysis of linegraphs
- Just like we've already seen with previous analysis, trends are almost always holding between detectors
- There is some potential noise here:
    - To get the most accurate graph, I would need to plot the x-axis by timestamp
        - Plotting with timestamp makes for results that are very hard to interpret since observations can be 4 seconds apart or 18 hours apart
    - I grouped them by `group` and plotted each group's mean instead
        - This data is full of outliers, but mean has to do.
            - Median and mode both graph 0 for almost all of `xray2_ps` and `xray3_ps`
        - Grouping and plotting by mean may allow for outliers to skew data, but we can see similar trends emerge in each detector anyway

# Get ratio of all detector combinations

In [ ]:
detectors = ["xray0", "xray1", "xray2", "xray3"]

fig, axes = plt.subplots(4, 4, figsize=(20, 20))

grouped = clean_rad.groupby("group")[["xray0", "xray1", "xray2", "xray3"]].mean()

for i, detector in enumerate(detectors):
    for j, detector2 in enumerate(detectors):
        axes[i, j].plot(grouped.index, grouped[detector]/grouped[detector2])
        axes[i, j].set_yscale('log')
        axes[i, j].set_title(f"{detector}/{detector2}")
        axes[i, j].set_ylabel("Particle Counts Per Second (Log)")
        axes[i, j].set_xlabel("Group")

fig.suptitle("X-ray Detector Ratios", fontsize="xx-large")
plt.show()

## Analysis of detector ratios
One last test to show similar trends

- It appears that each X-ray sensor has an energy threshold 10 times higher than the previous detector
- As you move down the visual, the graphs stay a similar shape
- As you move from left to right on the visual, graphs stay a similar shape but develop more holes
    - These holes develop because dividing by the `xray2_ps` and `xray3_ps` columns start to involve a lot of divisions by 0.
- These shapes staying the same between ratios shows yet again that there are similar trends between the four X-ray detectors

# Compare ratios for the electron and proton sensors
The above visual shows that every xray detector's threshold is 10 higher than the previous sensor. This made me curious if that's the case for proton0 and electron0 as well

In [ ]:
detectors = ["proton0", "proton1"]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

grouped = clean_rad.groupby("group")[["proton0", "proton1"]].mean()

for i, detector in enumerate(detectors):
    for j, detector2 in enumerate(detectors):
        axes[i, j].plot(grouped.index, grouped[detector]/grouped[detector2])
        axes[i, j].set_yscale('log')
        axes[i, j].set_title(f"{detector}/{detector2}")
        axes[i, j].set_ylabel("Particle Counts Per Second (Log)")
        axes[i, j].set_xlabel("Group")

fig.suptitle("Proton Detector Ratios", fontsize="xx-large")
plt.show()

#### It looks like proton 0's threshold is 100x lower than proton 1

In [ ]:
detectors = ["electron0", "electron1"]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

grouped = clean_rad.groupby("group")[["electron0", "electron1"]].mean()

for i, detector in enumerate(detectors):
    for j, detector2 in enumerate(detectors):
        axes[i, j].plot(grouped.index, grouped[detector]/grouped[detector2])
        axes[i, j].set_yscale('log')
        axes[i, j].set_title(f"{detector}/{detector2}")
        axes[i, j].set_ylabel("Particle Counts Per Second (Log)")
        axes[i, j].set_xlabel("Group")

fig.suptitle("Proton Detector Ratios", fontsize="xx-large")
plt.show()

#### It looks like electron 0's threshold is 10x lower than electron 1

# Final Analysis of X-ray sensors

In this file, my goal was to show that `xray0_ps` (and thus `xray0`) holds reliable data. We cannot prove to you that `xray0` has accurate data without sending up more satellites to make tests, but we were able to show similar trends between `xray0_ps`, `xray1_ps`, `xray2_ps`, and `xray3_ps`. This means that you can trust the results of `xray0` just as much as you can trust the results of the other three sensors. They either all work, or they are all failing in the same way.